In [25]:
import numpy as np 
import pandas as pd
from dataclasses import dataclass ## for defining datatypes remove the need for defining classes with init functions ##
from typing import List, Dict ## for defining datatypes for lists and dictionaries ##


 ### defining datatypes  ###
@dataclass
class Job:
    id: int
    workload: float
 
@dataclass 
class Node:
    id: int
    speed: float
    cost_per_hour: float
    
    def calculate_time(self, job: Job):
        return job.workload / self.speed
    
    
    

In [26]:

    
def generate_cloud_environment(num_jobs=50):
    nodes = [
        Node(id=1, speed=100, cost_per_hour=0.02),
        Node(id=2, speed=250, cost_per_hour=0.08),
        Node(id=3, speed=500, cost_per_hour=0.15)
    ]

    np.random.seed(42)
    mu, sigma = 6, 1.2  ## the mean and standard deviation for workload ##

    workloads = np.random.lognormal(mean=mu, sigma=sigma, size=num_jobs)  ## generating workloads using lognormal distribution ##
    workload = np.clip(workloads, a_min=10, a_max=None) 

    jobs = [Job(id=i, workload=workload[i]) for i in range(num_jobs)] 

    return jobs, nodes 


jobs, nodes = generate_cloud_environment(num_jobs=10)

print("Nodes:")
for node in nodes:
    print(f"Node ID: {node.id}, Speed: {node.speed}, Cost per Hour: ${node.cost_per_hour}")

print("\nJobs:")    
for job in jobs:
    print(f"Job ID: {job.id}, Workload: {job.workload:.2f}")

Nodes:
Node ID: 1, Speed: 100, Cost per Hour: $0.02
Node ID: 2, Speed: 250, Cost per Hour: $0.08
Node ID: 3, Speed: 500, Cost per Hour: $0.15

Jobs:
Job ID: 0, Workload: 732.20
Job ID: 1, Workload: 341.75
Job ID: 2, Workload: 877.63
Job ID: 3, Workload: 2508.99
Job ID: 4, Workload: 304.61
Job ID: 5, Workload: 304.61
Job ID: 6, Workload: 2683.98
Job ID: 7, Workload: 1013.25
Job ID: 8, Workload: 229.67
Job ID: 9, Workload: 773.61


In [27]:
def evaluate_schedule(jobs: List[Job], nodes: List[Node], assignments: List[int]) -> dict:
    """
    Takes a 1D array of assignments and returns the makespan and cost.
    Example assignments: [1, 3, 2] means:
    - job 0 is assigned to node 1
    - job 1 is assigned to node 3
    - job 2 is assigned to node 2

    Args:
        jobs: List of Job objects.
        nodes: List of Node objects.
        assignments: List of node IDs corresponding to each job's assigned node.

    Returns:
        A dictionary with schedule metrics.
    """
    # 1. Create empty queues for each node (using the Node's actual ID)
    node_queues = {node.id: [] for node in nodes}

    # 2. Route the jobs to their assigned nodes
    for job_idx, assigned_node_id in enumerate(assignments):
        current_job = jobs[job_idx]
        node_queues[assigned_node_id].append(current_job)

    total_cost = 0.0
    node_completion_times = {}

    # 3. Process each node's queue
    for node in nodes:
        queue = node_queues[node.id]

        # --- THE LOCAL HEURISTIC (Shortest Job First) ---
        # Sort the queue locally so the smallest workloads run first
        queue.sort(key=lambda j: j.workload)

        node_active_time = 0.0

        # Execute the jobs
        for job in queue:
            execution_time = job.workload / node.speed
            node_active_time += execution_time

        # Calculate cost for this specific node (Convert seconds to hours)
        node_cost = (node_active_time / 3600) * node.cost_per_hour
        total_cost += node_cost

        # Record how long this node took to finish all its work
        node_completion_times[node.id] = node_active_time

    # 4. The Makespan is the time of the node that finishes absolutely last
    makespan = max(node_completion_times.values()) if node_completion_times else 0.0

    return {
        "makespan_seconds": makespan,
        "total_cost_dollars": total_cost,
        "node_details": node_completion_times
    }

In [28]:
# Extract the valid node IDs (1, 2, and 3 from our previous cell)
available_node_ids = [n.id for n in nodes]

# Generate a random schedule (The "Chromosome")
# e.g., randomly assigning each of the 10 jobs to Node 1, 2, or 3
np.random.seed(42) # Keeping it reproducible
random_schedule = np.random.choice(available_node_ids, size=len(jobs))

print(f"Generated Random Schedule (Array):")
print(random_schedule)
print("-" * 40)

# Run the random schedule through our engine
results = evaluate_schedule(jobs, nodes, random_schedule)

print("--- Simulation Results (Random Baseline) ---")
print(f"Makespan (Total Time):   {results['makespan_seconds']:.2f} seconds")
print(f"Total Cloud Cost:        ${results['total_cost_dollars']:.4f}")
print("\nIndividual Node Completion Times:")
for node_id, time_spent in results['node_details'].items():
    print(f"  Node {node_id}: {time_spent:.2f} seconds")

Generated Random Schedule (Array):
[3 1 3 3 1 1 3 2 3 3]
----------------------------------------
--- Simulation Results (Random Baseline) ---
Makespan (Total Time):   15.61 seconds
Total Cloud Cost:        $0.0008

Individual Node Completion Times:
  Node 1: 9.51 seconds
  Node 2: 4.05 seconds
  Node 3: 15.61 seconds


In [29]:
import numpy as np

class GeneticAlgorithm:
    
    def __init__(self, jobs, nodes, pop_size=20, generations=100, mutation_rate=0.02, crossover_rate=0.95):
        self.jobs = jobs
        self.nodes = nodes
        self.pop_size = pop_size
        self.generations = generations
        self.mutation_rate = mutation_rate
        self.crossover_rate = crossover_rate
        
        self.valid_node_ids = [n.id for n in nodes]
        self.num_jobs = len(jobs)

    def initialize_population(self):
        """Creates a list of random schedules (1D arrays)."""
        return [np.random.choice(self.valid_node_ids, size=self.num_jobs) for _ in range(self.pop_size)]

    def calculate_fitness(self, schedule):
        """
        Our custom Multi-Objective Fitness Function (Makespan + Cost)
        This is an improvement over the paper's Makespan-only fitness function!
        """
        results = evaluate_schedule(self.jobs, self.nodes, schedule)
        fitness_score = results['makespan_seconds'] + (results['total_cost_dollars'] * 100)
        return fitness_score

    def tournament_selection(self, population, fitnesses, k=3):
        selected_indices = np.random.choice(len(population), size=k)
        best_idx = selected_indices[np.argmin([fitnesses[i] for i in selected_indices])]
        return population[best_idx]

    def crossover(self, p1, p2):
        point = np.random.randint(1, self.num_jobs - 1)
        c1 = np.concatenate([p1[:point], p2[point:]])
        c2 = np.concatenate([p2[:point], p1[point:]])
        return c1, c2

    def mutate(self, schedule):
        if np.random.rand() < self.mutation_rate:
            mutation_point = np.random.randint(self.num_jobs)
            new_node = np.random.choice(self.valid_node_ids)
            schedule[mutation_point] = new_node
        return schedule

    def run(self):
        population = self.initialize_population()
        best_overall_schedule = None
        best_overall_fitness = float('inf')
        fitness_history = [] 

        print(f"Starting GA Optimization for {self.generations} generations...")
        
        for gen in range(self.generations):
            fitnesses = [self.calculate_fitness(ind) for ind in population]

            min_idx = np.argmin(fitnesses)
            if fitnesses[min_idx] < best_overall_fitness:
                best_overall_fitness = fitnesses[min_idx]
                best_overall_schedule = population[min_idx]

            fitness_history.append(best_overall_fitness)

            new_pop = []
            new_pop.append(best_overall_schedule) 

            while len(new_pop) < self.pop_size:
                p1 = self.tournament_selection(population, fitnesses)
                p2 = self.tournament_selection(population, fitnesses)
                
                
                
                if np.random.rand() < self.crossover_rate:
                    c1, c2 = self.crossover(p1, p2)
                else:
                    c1, c2 = np.copy(p1), np.copy(p2)
                
                new_pop.append(self.mutate(c1))
                if len(new_pop) < self.pop_size:
                    new_pop.append(self.mutate(c2))

            population = new_pop

        return best_overall_schedule, fitness_history

In [30]:
ga = GeneticAlgorithm(jobs, nodes, pop_size=50, generations=100, mutation_rate=0.2)

best_ai_schedule, learning_history = ga.run()

ai_results = evaluate_schedule(jobs, nodes, best_ai_schedule)


print(f"Final Optimized Schedule Array:\n{best_ai_schedule}\n")

print("--- Final Metrics (AI vs Random Baseline) ---")
print(f"AI Makespan: {ai_results['makespan_seconds']:.2f} sec")
print(f"AI Cost:     ${ai_results['total_cost_dollars']:.4f}")

print("\nAI Node Utilization (Notice how it balances the load):")
for node_id, time_spent in ai_results['node_details'].items():
    print(f"  Node {node_id}: {time_spent:.2f} seconds")

Starting GA Optimization for 100 generations...
Final Optimized Schedule Array:
[3 2 1 2 3 3 3 3 1 3]

--- Final Metrics (AI vs Random Baseline) ---
AI Makespan: 11.62 sec
AI Cost:     $0.0008

AI Node Utilization (Notice how it balances the load):
  Node 1: 11.07 seconds
  Node 2: 11.40 seconds
  Node 3: 11.62 seconds


In [31]:
def greedy_sjf_schedule(jobs: List[Job], nodes: List[Node]) -> List[int]:
    """
    A deterministic 'smart' baseline. 
    Sorts all jobs smallest to largest, then assigns each to the node 
    that will finish it the soonest based on current loads.
    """
    # 1. Sort jobs by workload (smallest first) while keeping their original index
    sorted_jobs = sorted(enumerate(jobs), key=lambda x: x[1].workload)
    
    # Empty array to hold our final answers
    assignments = [0] * len(jobs)
    
    # Track the current active time of each node
    node_current_times = {n.id: 0.0 for n in nodes}
    
    # 2. Assign each job one by one
    for original_idx, job in sorted_jobs:
        best_node_id = None
        earliest_finish_time = float('inf')
        
        # Check which node will finish this job the earliest
        for node in nodes:
            execution_time = node.calculate_time(job)
            potential_finish_time = node_current_times[node.id] + execution_time
            
            if potential_finish_time < earliest_finish_time:
                earliest_finish_time = potential_finish_time
                best_node_id = node.id
                
        # 3. Lock in the assignment and update the node's total time
        assignments[original_idx] = best_node_id
        
        # We must add the execution time to that node's running total
        best_node = next(n for n in nodes if n.id == best_node_id)
        node_current_times[best_node_id] += best_node.calculate_time(job)
        
    return assignments

In [32]:
import pandas as pd

def get_advanced_metrics(results, jobs, nodes):
    makespan = results['makespan_seconds']
    num_nodes = len(nodes)
    
    # 1. Resource Utilization (%)
    total_busy_time = sum(results['node_details'].values())
    utilization = (total_busy_time / (makespan * num_nodes)) * 100 if makespan > 0 else 0
    
    # 2. Speedup (T_serial / T_parallel)
    fastest_node = max(nodes, key=lambda n: n.speed)
    serial_time = sum([job.workload / fastest_node.speed for job in jobs])
    speedup = serial_time / makespan if makespan > 0 else 0
    
    # 3. Efficiency (Speedup / N)
    efficiency = speedup / num_nodes if num_nodes > 0 else 0
    
    return utilization, speedup, efficiency



baseline_results = evaluate_schedule(jobs, nodes, random_schedule)
sjf_schedule = greedy_sjf_schedule(jobs, nodes)
sjf_results = evaluate_schedule(jobs, nodes, sjf_schedule)

rand_util, rand_speed, rand_eff = get_advanced_metrics(baseline_results, jobs, nodes)
sjf_util, sjf_speed, sjf_eff = get_advanced_metrics(sjf_results, jobs, nodes)
ai_util, ai_speed, ai_eff = get_advanced_metrics(ai_results, jobs, nodes)

comparison_data = {
    "Algorithm": [
        "1. Random (Baseline)", 
        "2. Pure SJF", 
        "3. GA"
    ],
    "Makespan (s)": [
        baseline_results['makespan_seconds'], 
        sjf_results['makespan_seconds'], 
        ai_results['makespan_seconds']
    ],
    "Cost ($)": [
        baseline_results['total_cost_dollars'], 
        sjf_results['total_cost_dollars'], 
        ai_results['total_cost_dollars']
    ],
    "Utilization (%)": [rand_util, sjf_util, ai_util],
    "Speedup": [rand_speed, sjf_speed, ai_speed],
    "Efficiency": [rand_eff, sjf_eff, ai_eff]
}

df = pd.DataFrame(comparison_data)

print(df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))


           Algorithm  Makespan (s)  Cost ($)  Utilization (%)  Speedup  Efficiency
1. Random (Baseline)       15.6122    0.0008          62.2908   1.2516      0.4172
         2. Pure SJF       15.0488    0.0008          59.9494   1.2985      0.4328
               3. GA       11.6245    0.0008          97.7831   1.6810      0.5603
